In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

# ====== [1] CSV에서 데이터 읽기 ======
df = pd.read_excel("./shops.xlsx")  # 파일명 맞게 수정
names = df['name'].astype(str).tolist()
categories = df['category'].astype(str).tolist()


# ====== [2] 카테고리 인덱싱 ======
unique_categories = sorted(set(categories))
cat2idx = {c: i for i, c in enumerate(unique_categories)}
idx2cat = {i: c for c, i in cat2idx.items()}

# ====== [3] 문자 사전 만들기 ======
chars = sorted(set("".join(names)))
char2idx = {ch: i+1 for i, ch in enumerate(chars)}  # 0은 padding
vocab_size = len(char2idx) + 1

# ====== [4] 인코딩 함수 ======
def encode_name(name, max_len=20):
    idxs = [char2idx.get(ch, 0) for ch in name]
    if len(idxs) < max_len:
        idxs += [0] * (max_len - len(idxs))
    return torch.tensor(idxs[:max_len])

X = torch.stack([encode_name(n) for n in names])
y = torch.tensor([cat2idx[c] for c in categories])

# ====== [5] 모델 정의 ======
class ShopClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embed(x)
        _, (h, _) = self.lstm(x)
        out = self.fc(h[-1])
        return out

# ====== [6] 학습 ======
model = ShopClassifier(vocab_size, 32, 64, len(unique_categories))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(50):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss {loss.item():.4f}")

# ====== [7] 테스트 ======
test_name = "연세대학교GS24"
test_encoded = encode_name(test_name)
pred = model(test_encoded.unsqueeze(0))
pred_cat = idx2cat[pred.argmax().item()]
print(f"입력: {test_name} → 예측 결과: {pred_cat}")

Epoch 0, Loss 1.6260
Epoch 10, Loss 1.5979
Epoch 20, Loss 1.5514
Epoch 30, Loss 1.4503
Epoch 40, Loss 1.2060
입력: 연세대학교GS24 → 예측 결과: 식비


In [19]:
# ====== [7] 테스트 ======
test_name = "컴포즈커피사당동점"
test_encoded = encode_name(test_name)
pred = model(test_encoded.unsqueeze(0))
pred_cat = idx2cat[pred.argmax().item()]
print(f"입력: {test_name} → 예측 결과: {pred_cat}")

입력: 컴포즈커피사당동점 → 예측 결과: 카페
